In [0]:
%pip install mlflow>=3.0 databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.7/774.7 kB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 137.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 138.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.9/609.9 kB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 146.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 1

In [0]:
import os
import mlflow
import mlflow.spark

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.sql.types import NumericType
from pyspark.ml.feature import Imputer, FeatureHasher, StandardScaler
from pyspark.ml.classification import LogisticRegression
from mlflow.models.signature import infer_signature
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from mlflow.tracking import MlflowClient

# -------------------------
# Config
# -------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS mlops_project")

SILVER_TBL = "mlops_project.lendingclub_silver"   # <- use your final Silver table
EXPERIMENT_PATH = "/Users/angelina.suchkova@mastercard.com/LendingClub_MLOps"

MODEL_NAME = "lr_end_to_end_pipeline_model"
ALIAS_NAME = "Champion"

HASH_DIM = 2**15
FIT_FRACTION = 0.1   # keep to avoid serverless/Spark Connect model-size issues

# UC temp dir for serverless/shared (safe)
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/mlops_project/mlflow_tmp")
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlops_project/mlflow_tmp"

# Config
# -------------------------
label_col = "label_default"
model_name = "lr_end_to_end_featurehasher_model"
experiment_path = "/Users/angelina.suchkova@mastercard.com/LendingClub_MLOps"


mlflow.set_experiment(EXPERIMENT_PATH)

label_col = "label_default"

# -------------------------
# Load data
# -------------------------
df = spark.table(SILVER_TBL)

# Splits
train_df, val_df, test_df = df.randomSplit([0.7, 0.15, 0.15], seed=42)

# Column lists
numeric_cols = [
    c for c, t in df.dtypes
    if t in ("int", "bigint", "double", "float") and c != label_col
]
categorical_cols = [c for c, t in df.dtypes if t == "string"]

# Drop leakage-ish categoricals (same logic you used)
drop_feature_cols = [c for c in df.columns if c.startswith("hardship_")] + ["pymnt_plan","emp_length", "title"]
categorical_cols_v2 = [c for c in categorical_cols if c not in drop_feature_cols]

print("Dropped categoricals:", [c for c in categorical_cols if c in drop_feature_cols])
print("Categoricals used:", len(categorical_cols_v2))

# -------------------------
# Class weights on TRAIN
# -------------------------
counts = train_df.groupBy(label_col).count().collect()
cnt = {r[label_col]: r["count"] for r in counts}
n0, n1 = cnt.get(0, 0), cnt.get(1, 0)
total = n0 + n1

w0 = total / (2.0 * n0) if n0 else 1.0
w1 = total / (2.0 * n1) if n1 else 1.0

train_w = train_df.withColumn(
    "class_weight",
    F.when(F.col(label_col) == 1, F.lit(w1)).otherwise(F.lit(w0))
)

# -------------------------
# End-to-end pipeline
# -------------------------
imputed_numeric_cols = [f"{c}__imputed" for c in numeric_cols]

imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=imputed_numeric_cols,
    strategy="median"
)

hasher = FeatureHasher(
    inputCols=categorical_cols_v2 + imputed_numeric_cols,
    outputCol="features",
    numFeatures=HASH_DIM
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withStd=True,
    withMean=False
)

lr = LogisticRegression(
    featuresCol="features_scaled",
    labelCol=label_col,
    weightCol="class_weight",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

pipeline = Pipeline(stages=[imputer, hasher, scaler, lr])

# -------------------------
# Train on sampled TRAIN (to avoid serverless issues)
# -------------------------
train_fit_df = train_w.sample(fraction=FIT_FRACTION, seed=42)

evaluator_auc = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

with mlflow.start_run(run_name="train_lr_end_to_end") as run:
    fitted = pipeline.fit(train_fit_df)

    # Validate
    val_scoring = val_df.withColumn("class_weight", F.lit(1.0))
    val_pred = fitted.transform(val_scoring)
    val_auc = evaluator_auc.evaluate(val_pred)

    # Signature
    input_example_pd = train_w.select("class_weight", label_col).limit(20).toPandas()
    pred_example_pd = val_pred.select("probability", "prediction").limit(20).toPandas()
    signature = infer_signature(input_example_pd, pred_example_pd)

    # Log params/metrics
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("encoding", "FeatureHasher")
    mlflow.log_param("numFeatures_hash", 2**18)
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("regParam", 0.0)
    mlflow.log_param("elasticNetParam", 0.0)
    mlflow.log_param("class_weighting", True)
    mlflow.log_metric("val_auc", float(val_auc))


    # Log + register
    model_info = mlflow.spark.log_model(
        spark_model=fitted,
        artifact_path="model",
        signature=signature,
        registered_model_name=MODEL_NAME
    )

    run_id = run.info.run_id
    print("Run ID:", run_id)
    print("Validation AUC:", val_auc)






2026/01/05 12:57:21 INFO mlflow.tracking.fluent: Experiment with name '/Users/angelina.suchkova@mastercard.com/LendingClub_MLOps' does not exist. Creating a new experiment.


Dropped categoricals: ['emp_length']
Categoricals used: 13


/local_disk0/.ephemeral_nfs/envs/pythonEnv-484e207a-3da2-4306-8871-fed1f1d56c71/lib/python3.12/site-packages/mlflow/system_metrics/metrics/gpu_monitor.py:11: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
2026/01/05 12:57:24 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-484e207a-3da2-4306-8871-fed1f1d56c71/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) th

Uploading artifacts:   0%|          | 0/40 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.lr_end_to_end_pipeline_model': https://dbc-8dfd01aa-fa50.cloud.databricks.com/explore/data/models/workspace/default/lr_end_to_end_pipeline_model/version/1?o=7474657027313099
2026/01/05 12:58:39 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/01/05 12:58:39 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Run ID: 92a02e9f82d349c6aab577d2f7a1b2d5
Validation AUC: 0.7538836336118666


In [0]:
# Promote alias automatically (Champion -> newest version)
# -------------------------
client = MlflowClient()
FULL_MODEL_NAME = f'workspace.default.{MODEL_NAME}'
# Find the newest version for this run
versions = client.search_model_versions(f"name='{FULL_MODEL_NAME}'")
versions_for_run = [v for v in versions if v.run_id == run_id]
if not versions_for_run:
    raise RuntimeError("Could not find registered model version for this run_id")

new_version = max(int(v.version) for v in versions_for_run)

client.set_registered_model_alias(FULL_MODEL_NAME, ALIAS_NAME, str(new_version))
print(f"Set alias '{ALIAS_NAME}' -> version {new_version} for model {FULL_MODEL_NAME}")

Set alias 'Champion' -> version 1 for model workspace.default.lr_end_to_end_pipeline_model
